```mermaid
graph LR
 日本語+Pythonのコーパス作成 --> トークナイザーの学習
 トークナイザーの学習 --> GPTモデルの設計
  GPTモデルの設計 --> 学習ループの実装
  学習ループの実装 --> ファインチューニング
  ファインチューニング --> 推論
```

| Step | Name |Description |
| ---- | ----------- |----------- |
| 1 | 日本語+Pythonのコーパス作成 | (1)日本語(20~200MB) --> Wikipedia, 青空文庫, ニュース系コーパス, (2)Python(20~200MB) --> GitHub, Kaggle Notebooks |
|2 | トークナイザーの学習 | 語彙数(16k~32k), 日本語(BPE < Unigram), Python code(インデントや記号を分離しすぎない) |
|3| GPTモデルの設計 | Embedding(token embedding, position embedding), Transformer Block(Multi-Head Attention, MLP, Layer Normalization, Residual), Language Model Heads(Softmax) |
|4| 学習ループの実装 | batch size(16~48), sequence length(256~512), learning time(1h~1day) |
|5| ファインチューニング | Kaggle Notebooks, Python code, Python Q&A |
|6| 推論 ||

In [ ]:
from datasets import load_dataset
import pandas as pd
import os
import spacy

In [ ]:
# Japanese datasets 
ds_wiki = load_dataset("mini97/filtered_japanese-wikipedia") 
ds_aozora = load_dataset("globis-university/aozorabunko-clean") 

# Python datasets 
ds_python = load_dataset("Arjun-G-Ravi/Python-codes")

In [ ]:
print(type(ds_wiki))
print(type(ds_aozora))
print(type(ds_python))

In [ ]:
ds_wiki.column_names

In [ ]:
ds_wiki["train"][0]

In [ ]:
ds_aozora.column_names

In [ ]:
ds_aozora["train"][0]

In [ ]:
ds_python.column_names

In [ ]:
ds_python["train"][0]

In [ ]:
ds_wiki.num_rows

In [ ]:
ds_aozora.num_rows

In [ ]:
ds_python.num_rows

In [ ]:
path = "data/corpus.txt"

os.makedirs(os.path.dirname(path), exist_ok=True)

if os.path.isfile(path):
    os.remove(path)

with open(path, "w"):
    pass

In [ ]:
def normalize(text):
    return text.replace("\n", " ")

def flatten_code(code: str) -> str:
    code = code.strip().replace("\r\n", "\n").replace("\n", " <NL> ")
    return code

# down sampling
ds_wiki_train = ds_wiki["train"].shuffle(seed=42).select(range(300000))

datasets = [
    (ds_wiki_train, "original"),
    (ds_aozora["train"], "text"),
    (ds_python["train"], ["code", "question"])
]

# buffer
BUFFER_SIZE = 1000

# load spacy model
nlp = spacy.load("ja_core_news_sm")

# 自然文(wiki, aozora)
buffer = []
with open(path, "w", encoding="utf-8") as f:
    for ds, column in datasets[:2]:
        for row in ds:
            text = row[column]
            doc = nlp(text)

            for sent in doc.sents:
                sentence = sent.text
                sentence = normalize(sentence)
                buffer.append(sentence)
                if len(buffer) == BUFFER_SIZE:
                    f.write("\n".join(buffer) + "\n")
                    buffer = []
    if buffer:
        f.write("\n".join(buffer) + "\n")
        buffer = []

# Python
buffer = []
with open(path, "a", encoding="utf-8") as f:
    for ds, columns in datasets[2:3]:
        code_col, q_col = columns

        for row in ds:
            code_text = row[code_col]
            q_text = row[q_col]
            combine = f"{code_text} {q_text}"
            combine = flatten_code(combine)
            combine = normalize(combine)
            buffer.append(combine)
            if len(buffer) == BUFFER_SIZE:
                f.write("\n".join(buffer) + "\n")
                buffer = []
    if buffer:
        f.write("\n".join(buffer) + "\n")
        buffer.clear()